# Phase 5: Attention Ablation Study (80 runs)
Coordinate Attention (CA) and Attention Gate on ResNet-50 and HybridNet.
Datasets: `pitt`, `italian_pd`, `physionet`, `pcgita` â€” 5 folds each.

In [ ]:
import os
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

KAGGLE_INPUT_DIR = Path('/kaggle/input')
WORK_DIR = Path('/kaggle/working/PROJECT')

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Check if Kaggle kept it as a zip or auto-extracted it
zips = list(KAGGLE_INPUT_DIR.glob('**/*.zip'))

if zips:
    ZIP_PATH = zips[0]
    print(f'Found zip: {ZIP_PATH}')
    print('Extracting... (may take 2-3 mins)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/kaggle/working/')
    print('Done extracting.')
else:
    print('No zip found â€” assuming Kaggle auto-extracted the dataset.')
    # Find the root of the dataset (where 'product' folder is)
    product_dirs = list(KAGGLE_INPUT_DIR.glob('**/product'))
    if not product_dirs:
        raise FileNotFoundError('Could not find the dataset contents!')
    
    src_dir = product_dirs[0].parent
    print(f'Copying files from {src_dir} to {WORK_DIR}...')
    
    # We copy them to working dir so we have write access
    shutil.copytree(src_dir, WORK_DIR, dirs_exist_ok=True)
    print('Done copying.')

In [ ]:
os.chdir(WORK_DIR)
print(f'CWD: {os.getcwd()}')

# Fix Windows backslash paths in all CSVs
ROOT = WORK_DIR
SPLITS = ROOT / 'product/artifacts/splits'
WIN_PREFIX = r'C:\FYP\PROJECT'

fixed = 0
for csv_file in SPLITS.glob('*.csv'):
    text = csv_file.read_text()
    if WIN_PREFIX in text or '\\' in text:
        text = text.replace(WIN_PREFIX, str(ROOT))
        text = text.replace('\\', '/')
        csv_file.write_text(text)
        fixed += 1
print(f'Fixed paths in {fixed} CSV files.')

In [ ]:
# Dynamically patch train_unified.py for the Attention Unfreeze bug
train_script_path = ROOT / 'product/training/train_unified.py'
content = train_script_path.read_text()

broken_logic = """    # Protocol: Transfer Learning Initial State
    for param in model.parameters():
        param.requires_grad = False
    
    # Always unfreeze head/gate
    if hasattr(model, 'alpha'):
        model.alpha.requires_grad = True
        for param in model.gate_bn.parameters():
            param.requires_grad = True"""

fixed_logic = """    # Protocol: Transfer Learning Initial State
    for param in model.parameters():
        param.requires_grad = False
        
    # Always unfreeze head/gate and injected attention modules
    attention_class_names = (
        'SEBlock', 'CBAM', 'CoordinateAttention', 
        'TripletAttention', 'SingleInputAttentionGate', 'AttentionGate'
    )
    for module in model.modules():
        if module.__class__.__name__ in attention_class_names:
            for param in module.parameters():
                param.requires_grad = True

    if hasattr(model, 'alpha'):
        model.alpha.requires_grad = True
    if hasattr(model, 'gate_bn'):
        for param in model.gate_bn.parameters():
            param.requires_grad = True"""

if broken_logic in content:
    content = content.replace(broken_logic, fixed_logic)
    train_script_path.write_text(content)
    print('Successfully patched train_unified.py with attention unfreeze fix!')
else:
    print('train_unified.py is already patched (or different signature)!')

In [ ]:
# Dynamically patch model_builder.py to fix the 16-module injection bug
builder_path = ROOT / 'product/models/model_builder.py'
if builder_path.exists():
    content = builder_path.read_text()
    # Patch ResNet injection
    broken_resnet = """    # Injection logic: Wrap every block in a stage
    for layer_name in ['layer1', 'layer2', 'layer3', 'layer4']:
        layer = getattr(model, layer_name)
        new_blocks = []
        for block in layer:
            if hasattr(block, 'conv3'):
                c = block.conv3.out_channels
            else:
                c = block.conv2.out_channels
            
            new_blocks.append(nn.Sequential(block, get_block(c)))
        
        setattr(model, layer_name, nn.Sequential(*new_blocks))"""
    
    fixed_resnet = """    # Injection logic: Apply a single attention module to the output of the final stage
    layer4 = model.layer4
    last_block = layer4[-1]
    
    if hasattr(last_block, 'conv3'):
        c = last_block.conv3.out_channels
    else:
        c = last_block.conv2.out_channels
        
    model.layer4 = nn.Sequential(*layer4, get_block(c))"""
    
    # Patch HybridNet injection
    broken_hybrid = """            # ResNet branch: wrap layer4 blocks with attention
            layer4 = model.layer4
            new_blocks = []
            for block in layer4:
                c = block.conv3.out_channels if hasattr(block, 'conv3') else block.conv2.out_channels
                new_blocks.append(nn.Sequential(block, _get_attention_block(attention_type, c)))
            model.layer4 = nn.Sequential(*new_blocks)"""
    
    fixed_hybrid = """            # ResNet branch: wrap layer4 blocks with attention
            layer4 = model.layer4
            last_block = layer4[-1]
            if hasattr(last_block, 'conv3'):
                c = last_block.conv3.out_channels
            else:
                c = last_block.conv2.out_channels
            model.layer4 = nn.Sequential(*layer4, _get_attention_block(attention_type, c))"""
            
    if broken_resnet in content:
        content = content.replace(broken_resnet, fixed_resnet)
        content = content.replace(broken_hybrid, fixed_hybrid)
        builder_path.write_text(content)
        print('Successfully patched model_builder.py to only inject ONE module!')
    else:
        print('model_builder.py is already patched!')


In [ ]:
def run_attention(dataset, model_type, fold, extra_args=None, epochs=15):
    """Run a single attention experiment, skip if already done."""
    run_id = f'{dataset}_{model_type}_fold{fold}'
    summary_path = ROOT / f'product/artifacts/runs/{dataset}/{run_id}/summary.json'
    
    if summary_path.exists():
        print(f'SKIP {run_id} (already done)')
        return
    
    print(f'>>> RUNNING {run_id}')
    cmd = [
        sys.executable, '-u', 'product/training/train_unified.py',
        '--dataset', dataset,
        '--model_type', model_type,
        '--fold', str(fold),
        '--epochs', str(epochs),
        '--batch_size', '32',
        '--drop_last',  # prevent single-sample batch crash
    ]
    if extra_args:
        cmd += extra_args
    
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    status = 'DONE' if process.returncode == 0 else 'ERROR'
    print(f'--- {status}: {run_id} ---')

In [ ]:
import shutil
# Clean old attention results so we re-run with new config
runs_root = ROOT / "product/artifacts/runs"
cleaned = 0
for ds in ["italian_pd", "physionet", "pcgita", "pitt"]:
    ds_dir = runs_root / ds
    if not ds_dir.exists():
        continue
    for run_dir in list(ds_dir.iterdir()):
        if any(tag in run_dir.name for tag in ["_ca_", "_gate_"]):
            shutil.rmtree(run_dir)
            cleaned += 1
print(f"Cleaned {cleaned} old attention runs.")


In [ ]:
# Strategy B: Fair Ablation — same config as Phase 3, just +attention
# Phase 3 used: lr=1e-4, unfreeze_at=0, no weighted_loss
# The ONLY difference is +/- attention module

T2_ARGS   = ["--lr", "1e-4", "--dropout", "0.5", "--unfreeze_at", "0"]
PITT_ARGS = ["--lr", "1e-5", "--dropout", "0.7", "--weight_decay", "0.1", "--unfreeze_at", "10"]

DATASET_ARGS = {
    "italian_pd": T2_ARGS,
    "physionet":  T2_ARGS,
    "pcgita":     T2_ARGS,
    "pitt":       PITT_ARGS,
}

ATTENTION_MODELS = ["resnet50_ca", "resnet50_gate", "hybrid_ca", "hybrid_gate"]
DATASETS = ["italian_pd", "physionet", "pcgita", "pitt"]

total = len(DATASETS) * len(ATTENTION_MODELS) * 5
done = 0

for dataset in DATASETS:
    extra_args = DATASET_ARGS[dataset]
    epochs = 30 if dataset == "pitt" else 15
    for model_type in ATTENTION_MODELS:
        for fold in range(5):
            run_attention(dataset, model_type, fold, extra_args, epochs=epochs)
            done += 1
            print(f"Progress: {done}/{total} runs complete")


In [ ]:
# Zip all results for download
output_zip = '/kaggle/working/phase5_attention_results.zip'
print(f'Zipping results...')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    runs_dir = ROOT / 'product/artifacts/runs'
    for f in runs_dir.glob('**/*'):
        if f.is_file() and not f.suffix == '.pt':  # skip big model weights
            zipf.write(f, f.relative_to(runs_dir.parent.parent))

print(f'Done! Download phase5_attention_results.zip from the Output tab.')